In [4]:
import csv
import json
import os
import random
from collections import defaultdict

# Dossier des fichiers CSV
DOSSIER_CSV = "Adjudication"
DOSSIER_SORTIE = "JSON_entrainementC1"
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

FICHIER_SORTIE_JSON = os.path.join(DOSSIER_SORTIE, "Corpus_Entrainement_Enrichi.json")
TAG_ID = {"PER": 1, "LOC": 2, "ORG": 3, "MISC": 4}

# Récupérer tous les fichiers CSV (consolidés et non consolidés)
fichiers = os.listdir(DOSSIER_CSV)
consolides = [f for f in fichiers if f.endswith("_consolide.csv")]
originaux = [f for f in fichiers if not f.endswith("_consolide.csv") and f.endswith(".csv")]

# Associer les consolidés à leur version originale
fichiers_associes = {}
for fichier in consolides:
    nom_base = fichier.replace("_consolide.csv", "")
    for original in originaux:
        if original.startswith(nom_base):
            fichiers_associes[nom_base] = (fichier, original)
            break

# Fonction pour traiter les fichiers et retourner la liste JSON
def traiter_fichiers(fichiers_associes, filtre_prefixes=None, pourcentage=None):
    donnees = []
    id_global = 1
    total_annotations = 0

    for nom_base, (fichier_consolide, fichier_original) in fichiers_associes.items():
        if filtre_prefixes and not any(prefix in fichier_consolide for prefix in filtre_prefixes):
            continue

        chemin_consolide = os.path.join(DOSSIER_CSV, fichier_consolide)
        chemin_original = os.path.join(DOSSIER_CSV, fichier_original)

        with open(chemin_consolide, newline='', encoding='utf-8') as f1, \
             open(chemin_original, newline='', encoding='utf-8') as f2:

            lecteur_consolide = list(csv.DictReader(f1, delimiter=';', quotechar='"'))
            lecteur_original = list(csv.DictReader(f2, delimiter=';', quotechar='"'))

            if pourcentage:
                taille = int(len(lecteur_consolide) * pourcentage)
                lecteur_consolide = lecteur_consolide[:taille]
                lecteur_original = lecteur_original[:taille]
                print(f"📊 {pourcentage*100:.0f}% de {nom_base} → {taille} lignes")

            tokens = []
            annotations = []
            position = 0

            for ligne_cons, ligne_orig in zip(lecteur_consolide, lecteur_original):
                token = ligne_cons['Token']
                label_maj = ligne_cons['Label_Maj'].strip()
                model_label = ligne_orig['Label'].strip()
                human_annotations = []

                for cle, valeur in ligne_orig.items():
                    if cle.endswith("_Correction") and valeur.strip() in TAG_ID:
                        human_annotations.extend([v.strip() for v in valeur.split(";") if v.strip() in TAG_ID])

                start_offset = position
                end_offset = position + len(token)

                if label_maj:
                    annotations.append({
                        "id": id_global,
                        "start_offset": start_offset,
                        "end_offset": end_offset,
                        "tag": {
                            "id": TAG_ID.get(label_maj, 99),
                            "name": label_maj
                        },
                        "tag_name": label_maj,
                        "model_annotations": [model_label] if model_label in TAG_ID else [],
                        "human_annotations": human_annotations
                    })
                    id_global += 1
                    total_annotations += 1

                tokens.append(token)
                position = end_offset + 1

            texte = " ".join(tokens)

            donnees.append({
                "id": nom_base,
                "content": texte,
                "metadata": {},
                "annotations": annotations
            })

            print(f"✅ Fichier traité : {fichier_consolide} ({len(annotations)} annotations)")

    return donnees, total_annotations

# ➤ Générer le fichier complet
print("\n📁 Création du fichier JSON complet")
donnees_finales, total_ann = traiter_fichiers(fichiers_associes)
with open(FICHIER_SORTIE_JSON, "w", encoding="utf-8") as f_json:
    json.dump(donnees_finales, f_json, indent=2, ensure_ascii=False)
print(f"✅ Fichier JSON enrichi créé : {FICHIER_SORTIE_JSON} ({total_ann} annotations)\n")

# ➤ Générer par type OCR
for prefix in ["REF", "Kraken", "Tesseract"]:
    print(f"\n📁 Création du fichier pour {prefix}")
    donnees, total_ann = traiter_fichiers(fichiers_associes, filtre_prefixes=[prefix])
    nom_sortie = os.path.join(DOSSIER_SORTIE, f"Corpus_{prefix}.json")
    with open(nom_sortie, "w", encoding="utf-8") as f_json:
        json.dump(donnees, f_json, indent=2, ensure_ascii=False)
    print(f"✅ Fichier {prefix} créé : {nom_sortie} ({total_ann} annotations)")

# ➤ Générer par pourcentage
for pct in [0.25, 0.5, 0.75]:
    print(f"\n📁 Création du fichier avec {int(pct*100)}% des données")
    donnees, total_ann = traiter_fichiers(fichiers_associes, pourcentage=pct)
    nom_sortie = os.path.join(DOSSIER_SORTIE, f"Corpus_{int(pct*100)}p.json")
    with open(nom_sortie, "w", encoding="utf-8") as f_json:
        json.dump(donnees, f_json, indent=2, ensure_ascii=False)
    print(f"✅ Fichier {int(pct*100)}% créé : {nom_sortie} ({total_ann} annotations)")


📁 Création du fichier JSON complet
✅ Fichier traité : adjudication_CARRAUD_petite-Jeanne_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv (380 annotations)
✅ Fichier traité : adjudication_CARRAUD_petite-Jeanne_REF_spaCy3.7.2-lg_consolide.csv (383 annotations)
✅ Fichier traité : adjudication_CARRAUD_petite-Jeanne_Kraken-base_spaCy3.7.2-lg 1_consolide.csv (389 annotations)
✅ Fichier traité : adjudication_DAUDET_petit-chose_Kraken-base_spaCy3.7.2-lg_consolide.csv (270 annotations)
✅ Fichier traité : adjudication_DAUDET_petit-chose_REF_spaCy3.7.2-lg_consolide.csv (280 annotations)
✅ Fichier traité : adjudication_DAUDET_petit-chose_TesseractFra-PNG_spaCy3.7.2-lg_consolide.csv (250 annotations)
✅ Fichier JSON enrichi créé : JSON_entrainementC1/Corpus_Entrainement_Enrichi.json (1952 annotations)


📁 Création du fichier pour REF
✅ Fichier traité : adjudication_CARRAUD_petite-Jeanne_REF_spaCy3.7.2-lg_consolide.csv (383 annotations)
✅ Fichier traité : adjudication_DAUDET_petit-chose_REF_spaCy3.7.2-